# Unleashed v1.2 — Python 3.13
Isi `BRANCH` untuk menguji branch lain (mis. `perf/phase1`); `main` = versi utama.
Buka UI lewat link **gradio.live** yang muncul di log sel *Jalankan Unleashed*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q --upgrade pip

In [ ]:
#@title Clone & install
BRANCH = "main"  #@param {type:"string"}
!git clone -b {BRANCH} https://github.com/antorio/unleashed.git /content/unleashed
%cd /content/unleashed
!git log --oneline -1
!mv config_colab.yaml config.yaml
!pip install -q -r requirements.txt python-multipart


In [ ]:
#@title Cek GPU (harus: RESULT: OK)
%cd /content/unleashed
!python tools/gpu_check.py


In [ ]:
#@title Jalankan Unleashed
# Buka UI lewat link gradio.live yang muncul di log di bawah.
# Hentikan: tombol stop sel ini.
# "Disconnect Colab when done" (tab Face Swap): setelah render, app menulis waktu
# disconnect ke FLAG; sel ini lalu menyinkronkan Google Drive (file hasil sampai
# di Drive), menghentikan app, dan memutus runtime. Gagal sinkron = tidak diputus.
from google.colab import userdata
from IPython.display import Pretty, display
import os, re, subprocess, sys, time
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'
FLAG = '/content/.unleashed_disconnect'
os.environ['UNLEASHED_DISCONNECT_FLAG'] = FLAG
if os.path.exists(FLAG):
    os.remove(FLAG)
LOG = '/content/run.log'
proc = subprocess.Popen([sys.executable, 'run.py'], cwd='/content/unleashed',
                        stdout=open(LOG, 'w'), stderr=subprocess.STDOUT)


def disconnect_due():
    try:
        with open(FLAG) as fh:
            return time.time() >= float(fh.read())
    except (OSError, ValueError):
        return False


def disconnect():
    from google.colab import drive, runtime
    os.remove(FLAG)
    print('\nRender selesai: menyinkronkan Google Drive ...', flush=True)
    try:
        drive.flush_and_unmount()
    except Exception as e:
        print(f'Sinkron Drive gagal ({e}): runtime TIDAK diputus.', flush=True)
        return
    print('Drive tersinkron. Menghentikan app dan memutus runtime.', flush=True)
    proc.terminate()
    runtime.unassign()


# Progress bar (tqdm) menimpa barisnya dengan \r. Log dibaca dari file (newline=''
# supaya \r tidak diubah jadi baris baru), dan \r ditangani di sini: baris yang
# masih ditimpa tampil sebagai SATU baris yang diperbarui di tempat, lalu tinggal
# sebagai keadaan akhirnya setelah \n.
live, rest, link = None, '', None


def show(text):
    global live, rest, link
    rest += text
    *done, rest = rest.split('\n')
    for line in done:
        line = line.rstrip('\r').rsplit('\r', 1)[-1]
        if live is not None:                # the line shown live is final now
            live.update(Pretty(line))
            live = None
        else:
            print(line, flush=True)
        if link is None:
            m = re.search(r'(https://\S+\.gradio\.live)', line)
            if m:
                link = m.group(1)
                print(f'\n>>> UI: {link}\n', flush=True)
    current = rest.rstrip('\r').rsplit('\r', 1)[-1]
    if current:
        if live is None:
            live = display(Pretty(current), display_id=True)
        else:
            live.update(Pretty(current))


log = open(LOG, errors='replace', newline='')
try:
    while proc.poll() is None:
        show(log.read())
        if disconnect_due():
            disconnect()
        time.sleep(1)
except KeyboardInterrupt:
    proc.terminate()
show(log.read() + '\n')

## Bandingkan dua hasil render (opsional)
Jalankan setelah server dihentikan. Isi dua path video hasil (mis. render `main` vs render branch dari klip & setting yang sama).

In [ ]:
#@title Compare renders
A = "/content/drive/MyDrive/c/hasil_main.mp4"  #@param {type:"string"}
B = "/content/drive/MyDrive/c/hasil_branch.mp4"  #@param {type:"string"}
!python tools/compare_renders.py "{A}" "{B}"


## Uji kestabilan render (opsional)
Hentikan dulu sel *Jalankan Unleashed*. Pilih uji yang diminta Claude, jalankan, lalu salin semua baris hasilnya ke Claude. Hanya menulis ke `/content/unleashed_test`.

In [ ]:
#@title Uji
TEST = "race_hunt"  #@param ["race_hunt", "repeat_render", "render_trace", "er_trace", "ort_race"]
%cd /content/unleashed
!git pull -q
!python tools/{TEST}.py